# 04 — Vocabulary, metadata tại thời điểm t và bộ mẫu prefix–target

Bước F của pipeline: biến mỗi phiên có ít nhất 2 nhóm timestamp thành các **query** (prefix `g1..gt`) và **target** (tập item phân biệt của `g(t+1)`).

- Đầu vào: `timestamp_groups.parquet`, `events_sessionized.parquet` (notebook 02); `item_property_history.parquet`, `item_catalog.parquet` (notebook 03); `configs/protocol.json`.
- Đầu ra:
  - `data/processed/item_vocab.parquet`: `itemid, item_idx` (D5: mọi item của train, `PAD=0`, `UNK=1`, item từ 2, sắp theo itemid).
  - `data/processed/samples/{split}_queries.parquet`: một dòng mỗi query.
  - `data/processed/samples/{split}_targets.parquet`: dạng dài, một dòng mỗi (query, item target).
  - `reports/samples/metrics.json`.
- Prefix **không lưu trùng**: đọc lại từ `events_sessionized` bằng `rr.samples.get_prefix()` / `prefix_events()`.
- Mọi số liệu được so với `reports/gru_readiness`; lệch thì dừng trước khi ghi.

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

Mỗi tệp đầu vào phải khớp SHA-256 ghi trong report của bước tạo ra nó.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, save_verified, utc_now, verify_inputs, write_json
from rr.samples import (BEHAVIORS, FIRST_IDX, PAD_IDX, UNK_IDX, asof_property, build_queries, build_targets,
                        build_vocab, encode, get_prefix, prefix_events)
from rr.candidates import CandidateFilter

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
SAMPLES = PROCESSED / "samples"
REPORT_PATH = ROOT / "reports/samples/metrics.json"
SPLITS = ["train", "validation", "test"]

split_report = read_json(ROOT / "reports/session_split/metrics.json")
metadata_report = read_json(ROOT / "reports/metadata/metrics.json")
expected_inputs = {}
for name in ["events_sessionized.parquet", "timestamp_groups.parquet"]:
    path = f"data/processed/{name}"
    expected_inputs[path] = split_report["artifacts"][path]
for name in ["item_property_history.parquet", "item_catalog.parquet"]:
    path = f"data/processed/{name}"
    expected_inputs[path] = metadata_report["artifacts"][path]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected_inputs.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| protocol", PROTOCOL["version"])

## 2. Ví dụ biên trên dữ liệu nhỏ tự tạo

Phiên 1: `g1 = view A`, `g2 = view B`, `g3 = addtocart B + view C` (cùng timestamp). Phiên 2 chỉ có một nhóm nên không sinh query.

- Query thứ hai có target `{B, C}`; B là target lặp (đã có trong prefix), C là target mới.
- Prefix không chứa nhóm target; metadata as-of không dùng bản cập nhật tương lai.

In [ ]:
_events = pd.DataFrame({
    "event_id": [0, 1, 2, 3, 4], "session_id": [1, 1, 1, 1, 2], "visitorid": [7, 7, 7, 7, 8],
    "group_id": [10, 11, 12, 12, 13], "group_index": [1, 2, 3, 3, 1], "timestamp": [100, 200, 300, 300, 50],
    "itemid": [65, 66, 66, 67, 65],
    "event": pd.Categorical(["view", "view", "addtocart", "view", "view"], categories=BEHAVIORS),
})
_groups = _events.drop_duplicates("group_id")[["group_id", "session_id", "visitorid", "timestamp", "group_index"]]
_q = build_queries(_groups, "train")
_t = build_targets(_q, _events)
assert len(_q) == 2 and _q["prefix_len_groups"].tolist() == [1, 2]
assert set(_t.loc[_t["query_id"] == 1, "itemid"]) == {66, 67}
assert _t.set_index(["query_id", "itemid"]).loc[(1, 66), "target_is_repeat"]
assert not _t.set_index(["query_id", "itemid"]).loc[(1, 67), "target_is_repeat"]
assert get_prefix(_events, _q.iloc[1])["group_id"].tolist() == [10, 11]
_v = build_vocab(_events[_events["session_id"] == 2])
assert _v["item_idx"].tolist() == [FIRST_IDX] and encode([65, 66], _v).tolist() == [FIRST_IDX, UNK_IDX]
_h = pd.DataFrame({"itemid": [67, 67], "timestamp": [150, 250], "property": ["categoryid"] * 2, "value": ["5", "9"]})
_asof = asof_property(pd.DataFrame({"itemid": [67, 67], "query_time_ms": [100, 200]}), _h, "categoryid")
assert _asof["categoryid_asof"].tolist() == [pd.NA, "5"]
del _events, _groups, _q, _t, _v, _h, _asof
print("Đã kiểm tra ví dụ biên: target tập item, cờ lặp, prefix, vocabulary/UNK, metadata as-of.")

## 3. Đọc dữ liệu và tạo vocabulary (D5)

Vocabulary = mọi `itemid` xuất hiện trong sự kiện `train`, kể cả phiên chỉ có một nhóm. `boundary_drop` không bao giờ được dùng.

In [ ]:
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["event_id", "session_id", "visitorid", "group_id", "group_index", "timestamp",
                                  "itemid", "event", "split"])
groups = pd.read_parquet(PROCESSED / "timestamp_groups.parquet",
                         columns=["group_id", "session_id", "visitorid", "timestamp", "group_index", "split"])
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
events["split"] = events["split"].astype(str)
groups["split"] = groups["split"].astype(str)

vocab = build_vocab(events[events["split"] == "train"])
vocab_rule = PROTOCOL["vocabulary"]
require(vocab_rule["pad_idx"] == PAD_IDX and vocab_rule["unk_idx"] == UNK_IDX and vocab_rule["first_item_idx"] == FIRST_IDX,
        "Mã PAD/UNK trong protocol khác rr.samples.")
require(vocab["itemid"].is_monotonic_increasing and vocab["itemid"].is_unique, "Vocabulary phải tăng dần, không trùng.")
print(f"Vocabulary train: {len(vocab):,} item (item_idx {vocab['item_idx'].min()}..{vocab['item_idx'].max()})")

## 4. Query và target cho từng tập

- `query_id` duy nhất trên cả ba tập (train bắt đầu từ 0, validation và test nối tiếp).
- `query_time_ms` = timestamp của nhóm cuối prefix; `target_time_ms` = timestamp của nhóm target.
- Target giữ **ID gốc**; `item_idx = UNK` nếu ngoài vocabulary; `in_vocab`, `candidate_eligible` (luật chính của protocol v1.2.0: trong vocabulary train và `known_at_train_ms <= t`) và `eligible_with_availability` (thêm luật `available ≠ 0`, chỉ dùng cho báo cáo phụ) được ghi riêng để báo cáo, **không loại khỏi mẫu số**.
- `categoryid_asof`: danh mục gần nhất có `timestamp <= query_time_ms`, thiếu thì `<NA>` (unknown).

In [ ]:
cand_filter = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
avail_filter = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms", use_availability=True)
queries, targets = {}, {}
next_id = 0
for split in SPLITS:
    q = build_queries(groups[groups["split"] == split], split, first_query_id=next_id)
    next_id += len(q)
    t = build_targets(q, events[events["split"] == split])
    t = t.merge(q[["query_id", "query_time_ms", "target_time_ms"]], on="query_id", validate="many_to_one")
    t["item_idx"] = encode(t["itemid"].to_numpy(), vocab)
    t["in_vocab"] = t["item_idx"] != UNK_IDX
    t["candidate_eligible"] = cand_filter.eligible(t["itemid"].to_numpy(), t["query_time_ms"].to_numpy())
    t["eligible_with_availability"] = avail_filter.eligible(t["itemid"].to_numpy(), t["query_time_ms"].to_numpy())
    asof = asof_property(t, history, "categoryid")
    t["categoryid_asof"] = asof["categoryid_asof"]
    t["categoryid_asof_ms"] = asof["categoryid_asof_ms"]
    t["has_strong_behavior"] = t["behaviors"].map(lambda b: "addtocart" in b or "transaction" in b)
    q_flags = t.groupby("query_id").agg(n_targets=("itemid", "size"), any_in_vocab=("in_vocab", "any"),
                                        any_repeat=("target_is_repeat", "any"), all_repeat=("target_is_repeat", "all"),
                                        target_has_strong=("has_strong_behavior", "any"))
    q = q.join(q_flags, on="query_id")
    q["n_targets"] = q["n_targets"].astype(np.int32)
    queries[split] = q
    targets[split] = t[["query_id", "itemid", "item_idx", "behaviors", "target_is_repeat", "has_strong_behavior",
                        "in_vocab", "candidate_eligible", "eligible_with_availability", "categoryid_asof", "categoryid_asof_ms"]].reset_index(drop=True)
    print(f"{split:>10}: {len(q):,} query, {len(t):,} lượt target")
display(queries["validation"].head())
display(targets["validation"].head())

## 5. Kiểm tra chống rò rỉ

- Mọi target có `target_time_ms > query_time_ms` và nhóm target là nhóm ngay sau prefix.
- Prefix (đọc lại từ `events_sessionized`) không chứa sự kiện của nhóm target và mọi sự kiện có `timestamp <= query_time_ms`. Kiểm tra toàn bộ query validation/test bằng `prefix_events`, và mẫu ngẫu nhiên bằng `get_prefix`.
- Metadata as-of không có `timestamp > query_time_ms`.

In [ ]:
leak = {}
group_time = groups.set_index("group_id")["timestamp"]
for split in SPLITS:
    q = queries[split]
    require((q["target_time_ms"] > q["query_time_ms"]).all(), f"{split}: target không nằm sau query.")
    require(group_time.reindex(q["target_group_id"]).to_numpy().tolist() == q["target_time_ms"].tolist(),
            f"{split}: target_group_id không khớp thời gian.")
    t = targets[split].merge(q[["query_id", "query_time_ms"]], on="query_id")
    known = t["categoryid_asof_ms"].notna()
    require((t.loc[known, "categoryid_asof_ms"] <= t.loc[known, "query_time_ms"]).all(), f"{split}: metadata từ tương lai.")
    leak[split] = {"targets_after_query": True, "metadata_not_from_future": True,
                   "targets_with_category_asof": int(known.sum())}

ev_eval = events[events["split"].isin(["validation", "test"])]
for split in ["validation", "test"]:
    q = queries[split]
    pe = prefix_events(q, ev_eval, columns=("itemid", "timestamp"), chunk=20_000)
    pe = pe.merge(q[["query_id", "query_time_ms"]], on="query_id")
    require((pe["timestamp"] <= pe["query_time_ms"]).all(), f"{split}: prefix có sự kiện sau query_time.")
    n_groups = pe.groupby("query_id")["group_index"].nunique()
    require(n_groups.reindex(q["query_id"]).tolist() == q["prefix_len_groups"].tolist(), f"{split}: prefix thiếu nhóm.")
    leak[split]["prefix_events_checked"] = len(pe)
    del pe

rng = np.random.default_rng(PROTOCOL["seed"])
by_session = events.set_index("session_id").sort_index()
sample = queries["train"].iloc[rng.choice(len(queries["train"]), 2000, replace=False)]
for _, row in sample.iterrows():
    prefix = get_prefix(by_session.loc[[row["session_id"]]].reset_index(), row)
    require(not prefix["group_id"].eq(row["target_group_id"]).any(), "Prefix chứa nhóm target.")
    require(prefix["timestamp"].max() == row["query_time_ms"], "Nhóm cuối prefix sai.")
leak["train"]["get_prefix_sample_checked"] = len(sample)
del by_session, ev_eval
display(pd.DataFrame(leak).T)

## 6. Đối chiếu với `reports/gru_readiness`

Số query, số lượt target, OOV theo vocabulary toàn train, query toàn target ngoài vocab và query có nhiều item target phải khớp khảo sát cũ.

In [ ]:
readiness = read_json(ROOT / "reports/gru_readiness/metrics.json")["splits"]
EXPECTED = {
    "queries": {s: readiness[s]["queries"] for s in SPLITS},
    "target_occurrences": {s: readiness[s]["distinct_target_item_occurrences"] for s in SPLITS},
    "multi_item_queries": {s: readiness[s]["queries_with_multiple_target_items"] for s in SPLITS},
    "repeat_target_occurrences": {s: readiness[s]["target_items_already_in_prefix"] for s in SPLITS},
    "oov_target_occurrences": {s: readiness[s]["target_coverage"]["all_train"]["unseen_target_occurrences"]
                               for s in ["validation", "test"]},
    "all_oov_queries": {s: readiness[s]["target_coverage"]["all_train"]["queries_with_all_targets_unseen"]
                        for s in ["validation", "test"]},
}
observed = {
    "queries": {s: len(queries[s]) for s in SPLITS},
    "target_occurrences": {s: len(targets[s]) for s in SPLITS},
    "multi_item_queries": {s: int((queries[s]["n_targets"] > 1).sum()) for s in SPLITS},
    "repeat_target_occurrences": {s: int(targets[s]["target_is_repeat"].sum()) for s in SPLITS},
    "oov_target_occurrences": {s: int((~targets[s]["in_vocab"]).sum()) for s in ["validation", "test"]},
    "all_oov_queries": {s: int((~queries[s]["any_in_vocab"]).sum()) for s in ["validation", "test"]},
}
rows = [(m, s, EXPECTED[m][s], observed[m][s]) for m in EXPECTED for s in EXPECTED[m]]
comparison = pd.DataFrame(rows, columns=["metric", "split", "expected", "observed"])
comparison["match"] = comparison["expected"] == comparison["observed"]
display(comparison)
require(comparison["match"].all(), "Số liệu lệch reports/gru_readiness; dừng để người dùng xem lại.")
print("Mọi tiêu chí khớp khảo sát cũ.")

## 7. Thống kê mô tả cho báo cáo

Tỷ lệ target lặp, target ngoài vocabulary, target không đủ điều kiện ứng viên tại `t`, target có hành vi mạnh (addtocart/transaction) và phân bố độ dài prefix. Đây là các con số cần nêu cạnh bảng kết quả mô hình.

In [ ]:
def describe(split):
    q, t = queries[split], targets[split]
    bins = pd.cut(q["prefix_len_groups"], [0, 1, 4, 9, np.inf], labels=["1", "2-4", "5-9", "10+"])
    return {
        "queries": len(q), "target_occurrences": len(t),
        "repeat_target_pct": 100 * float(t["target_is_repeat"].mean()),
        "queries_all_repeat_pct": 100 * float(q["all_repeat"].mean()),
        "oov_target_pct": 100 * float((~t["in_vocab"]).mean()),
        "not_candidate_eligible_pct": 100 * float((~t["candidate_eligible"]).mean()),
        "not_eligible_with_availability_pct": 100 * float((~t["eligible_with_availability"]).mean()),
        "queries_with_eligible_target_pct": 100 * float(
            t.groupby("query_id")["candidate_eligible"].any().mean()),
        "strong_behavior_target_pct": 100 * float(t["has_strong_behavior"].mean()),
        "queries_with_strong_target": int(q["target_has_strong"].sum()),
        "target_category_known_pct": 100 * float(t["categoryid_asof"].notna().mean()),
        "prefix_len_bins": {str(k): int(v) for k, v in bins.value_counts(sort=False).items()},
    }
stats = {s: describe(s) for s in SPLITS}
display(pd.DataFrame(stats).T)

## 8. Lưu Parquet đã kiểm tra và báo cáo

In [ ]:
artifacts = {}
info = save_verified(vocab, PROCESSED / "item_vocab.parquet", ["itemid"], ROOT)
artifacts[info.pop("path")] = info
for split in SPLITS:
    for frame, name, key in [(queries[split], f"{split}_queries.parquet", ["query_id"]),
                             (targets[split], f"{split}_targets.parquet", ["query_id", "itemid"])]:
        info = save_verified(frame, SAMPLES / name, key, ROOT)
        artifacts[info.pop("path")] = info
        print(f"Đã lưu {name}: {len(frame):,} dòng")

require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
report = {
    "generated_at_utc": utc_now(), **environment(),
    "notebook": "notebooks/04_vocabulary_va_bo_mau.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs,
    "vocabulary": {"items": len(vocab), "pad_idx": PAD_IDX, "unk_idx": UNK_IDX, "first_item_idx": FIRST_IDX,
                   "source": PROTOCOL["vocabulary"]["source"]},
    "query_id_ranges": {s: [int(queries[s]["query_id"].min()), int(queries[s]["query_id"].max())] for s in SPLITS},
    "expected_vs_observed": comparison.to_dict(orient="records"),
    "leakage_checks": leak,
    "stats": stats,
    "artifacts": artifacts,
    "status": "PASS",
}
write_json(REPORT_PATH, report)
print("Báo cáo:", rel(REPORT_PATH, ROOT))